# Car Auction Analysis
### Harsh Srivastava | IronLot Task 1


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

df = pd.read_csv('car_auction_train.csv')
print('dataset shape:', df.shape)
df.head()

In [ ]:
# checking what we're working with
print('column dtypes:')
print(df.dtypes)
print()
print('basic stats:')
df.describe()

## 1. Data Cleaning

first thing i noticed was a lot of missing values and some weird entries in the data

In [ ]:
# checking missing values
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

print('missing value summary:')
pd.DataFrame({'count': missing, 'percentage': missing_pct})[missing > 0]

In [ ]:
# transmission has 11.7% missing which is a lot
# looked at price distribution for auto vs manual vs unknown
# they have different median prices so Unknown is worth keeping as its own category
print('median price by transmission:')
print(df.groupby(df['transmission'].fillna('Unknown'))['sellingprice'].median())

In [ ]:
# cleaning the data
df_clean = df.copy()

# price filter
# below 500 are clearly wrong entries (saw prices like $1, $4 in data)
# above 150k are ultra luxury cars, very different market, less than 0.03% of data
before = len(df_clean)
df_clean = df_clean[(df_clean['sellingprice'] >= 500) & (df_clean['sellingprice'] <= 150000)]
print(f'price filter removed {before - len(df_clean)} rows')

# odometer filter
# values above 300k are sensor resets on old cars, not real readings
df_clean = df_clean[df_clean['odometer'] <= 300000]

# numeric imputation with median
# using median not mean because both columns have outliers
df_clean['odometer'].fillna(df_clean['odometer'].median(), inplace=True)
df_clean['condition'].fillna(df_clean['condition'].median(), inplace=True)

CAT_COLS = ['make', 'model', 'trim', 'body', 'transmission', 'color', 'interior', 'state']
for col in CAT_COLS:
    df_clean[col] = df_clean[col].fillna('Unknown')

print(f'final clean dataset: {len(df_clean)} rows')

## 2. EDA

tried to go beyond basic charts and find patterns that actually affect pricing

In [ ]:
# adding car_age for analysis
CURRENT_YEAR = 2015
df_clean['car_age'] = CURRENT_YEAR - df_clean['year']

# depreciation by body type
# interesting to see which body types hold value better
top_bodies = df_clean['body'].value_counts().head(5).index.tolist()
sub = df_clean[df_clean['body'].isin(top_bodies)]

fig, ax = plt.subplots(figsize=(10, 5))
for body in top_bodies:
    grp = sub[sub['body'] == body].groupby('car_age')['sellingprice'].median()
    ax.plot(grp.index, grp.values, marker='o', markersize=3, label=body)

ax.set_xlabel('Car Age (years)')
ax.set_ylabel('Median Selling Price')
ax.set_title('How Different Body Types Depreciate Over Time')
ax.legend()
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('depreciation_curves.png', dpi=150)
plt.show()

print('SUVs and Pickups hold value better than Sedans after age 3')

In [ ]:
# condition rating vs price
# wanted to check if its linear or if there are certain thresholds where price collapses

cond_stats = df_clean.groupby('condition')['sellingprice'].agg(['median', 'count'])
print(cond_stats)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(cond_stats.index, cond_stats['median'], color='steelblue', width=0.12)
ax.set_xlabel('Condition Rating')
ax.set_ylabel('Median Price')
ax.set_title('Condition vs Price - Price Drops Sharply Below Rating 2')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('condition_price.png', dpi=150)
plt.show()

# not a smooth linear relationship
# condition below 2 causes a steep drop, not gradual

In [ ]:
# odometer vs price for different age groups
# wanted to see if odo_per_year (usage intensity) is more useful than raw odometer

df_clean['odo_per_year'] = df_clean['odometer'] / df_clean['car_age'].replace(0, 0.5)
df_clean['usage_bin'] = pd.qcut(df_clean['odo_per_year'], q=5,
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

print('median price by usage intensity per year:')
usage_price = df_clean.groupby('usage_bin')['sellingprice'].median()
print(usage_price)
print(f'\nprice difference very low vs very high usage: ${usage_price.iloc[0] - usage_price.iloc[-1]:,.0f}')
print('odo_per_year is a stronger signal than raw odometer especially for newer cars')

In [ ]:
# transmission x body type
# checking which combinations beat the segment average price

segment_avg = df_clean.groupby('body')['sellingprice'].mean().rename('seg_avg')
combo = df_clean.groupby(['body', 'transmission'])['sellingprice'].mean().reset_index()
combo = combo.merge(segment_avg, on='body')
combo['vs_avg_pct'] = (combo['sellingprice'] / combo['seg_avg'] - 1) * 100

print('top combos that beat segment average:')
print(combo[combo['transmission'] != 'Unknown'].sort_values('vs_avg_pct', ascending=False).head(8).to_string(index=False))

## 3. Model Training and Results

In [ ]:
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error

# loading saved model and encoders
with open('model_Harsh.pkl', 'rb') as f:
    bundle = pickle.load(f)
model = bundle['model']
features = bundle['features']

with open('encoders_Harsh.pkl', 'rb') as f:
    encoders = pickle.load(f)

print('model loaded')
print('number of features:', len(features))
print('features:', features)

In [ ]:
# preparing features for validation
df_clean['condition_x_age'] = df_clean['condition'] * df_clean['car_age']
df_clean['low_odo_good_cond'] = (df_clean['odometer'] < 30000).astype(int) * df_clean['condition']

for col in CAT_COLS:
    le = encoders[col]
    df_clean[col + '_enc'] = df_clean[col].apply(
        lambda x: le.transform([x])[0] if x in le.classes_ else 0
    )

X = df_clean[features].values
y = df_clean['sellingprice'].values

_, X_val, _, y_val = train_test_split(X, y, test_size=0.15, random_state=42)

preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
mae = mean_absolute_error(y_val, preds)
r2 = model.score(X_val, y_val)

print(f'Validation RMSE : ${rmse:,.0f}')
print(f'Validation MAE  : ${mae:,.0f}')
print(f'R squared       : {r2:.4f}')

In [ ]:
# predicted vs actual scatter
fig, ax = plt.subplots(figsize=(7, 7))
idx = np.random.choice(len(y_val), 3000, replace=False)
ax.scatter(y_val[idx], preds[idx], alpha=0.3, s=8, color='steelblue')
ax.plot([0, 150000], [0, 150000], 'r--', lw=1, label='perfect prediction')
ax.set_xlabel('Actual Price')
ax.set_ylabel('Predicted Price')
ax.set_title(f'Predicted vs Actual  |  RMSE = ${rmse:,.0f}  |  R2 = {r2:.3f}')
ax.legend()
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
plt.tight_layout()
plt.savefig('pred_vs_actual.png', dpi=150)
plt.show()

In [ ]:
# feature importance
imp = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print('top 10 features:')
print(imp.head(10))

fig, ax = plt.subplots(figsize=(9, 5))
imp.head(12).plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title('Feature Importances')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

## 4. Bid Sequence Design

the bid is a pure function of predicted value V, bankroll B, current highest bid H, and round number R

**formula:**
```
discount(R) = 0.93  if R <= 3  (early, aggressive)
            = 0.90  if R <= 6  (mid)
            = 0.87  otherwise  (late, conservative)

max_bid = min(V x discount(R),  B x 0.20)

aggression(R) = 0.50  if R <= 3
              = 0.35  if R <= 6  
              = 0.20  otherwise

increment = max(100,  (max_bid - H) x aggression(R))
bid = round_to_50(H + increment)
bid = 0 if bid > max_bid  (pass)
```

**why this works better than flat increments:**
- early rounds we jump 50% of the gap which closes deals faster instead of dragging bidding wars
- we never pay more than 87-93% of predicted value so there's always a profit margin
- 20% bankroll cap per car means one bad prediction cant wipe out the budget
- late round patience (20% jumps) lets rivals overpay first before we commit

**aggression vs margin tradeoff:**
- higher discount (0.95+) = win more cars but thin margins, more risk
- lower discount (0.85) = fewer wins but each win is very profitable
- the blended approach aims for roughly 10% margin target while staying competitive